# Task 2: Loan Approval Prediction

**SAM AI Technologies — Data Science Internship**

Predict whether a loan application will be approved. Includes missing value handling, categorical encoding, model comparison, and evaluation via confusion matrix and accuracy.

Dataset: Analytics Vidhya Loan Prediction dataset (public mirror; falls back to synthetic data if offline).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

RANDOM_STATE = 42
OUT_DIR = "."

## 1. LOAD DATA

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/shrikant-temburwar/Loan-Prediction-Dataset/master/train.csv"

def load_data():
    try:
        df = pd.read_csv(DATA_URL)
        print(f"Loaded real dataset from {DATA_URL}  ({len(df)} rows)")
        return df
    except Exception as e:
        print(f"Could not download dataset ({e}). Generating synthetic fallback data instead.")
        rng = np.random.default_rng(RANDOM_STATE)
        n = 400
        df = pd.DataFrame({
            "Loan_ID": [f"LP{i:05d}" for i in range(n)],
            "Gender": rng.choice(["Male", "Female", None], n, p=[0.75, 0.23, 0.02]),
            "Married": rng.choice(["Yes", "No", None], n, p=[0.65, 0.33, 0.02]),
            "Dependents": rng.choice(["0", "1", "2", "3+", None], n, p=[0.55, 0.18, 0.15, 0.10, 0.02]),
            "Education": rng.choice(["Graduate", "Not Graduate"], n, p=[0.78, 0.22]),
            "Self_Employed": rng.choice(["Yes", "No", None], n, p=[0.14, 0.82, 0.04]),
            "ApplicantIncome": rng.integers(1500, 20000, n),
            "CoapplicantIncome": rng.integers(0, 8000, n),
            "LoanAmount": rng.integers(9, 500, n).astype(float),
            "Loan_Amount_Term": rng.choice([360, 180, 120, 60, None], n, p=[0.75, 0.1, 0.05, 0.05, 0.05]),
            "Credit_History": rng.choice([1.0, 0.0, np.nan], n, p=[0.75, 0.15, 0.10]),
            "Property_Area": rng.choice(["Urban", "Semiurban", "Rural"], n),
        })
        score = (df["Credit_History"].fillna(0) * 3 + (df["ApplicantIncome"] > 4000) * 1.0
                 + (df["Education"] == "Graduate") * 0.5 + rng.normal(0, 1, n))
        df["Loan_Status"] = np.where(score > np.nanmedian(score), "Y", "N")
        return df

df = load_data()

## 2. MISSING VALUES

In [ ]:
print("\n--- Missing values before cleaning ---")
missing = df.isnull().sum()
print(missing[missing > 0])

plt.figure(figsize=(8, 4))
missing[missing > 0].sort_values(ascending=False).plot(kind="bar", color="tomato")
plt.title("Missing Values by Column")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/loan_missing_values.png", dpi=120)
plt.close()
print("Saved loan_missing_values.png")

df = df.drop(columns=["Loan_ID"], errors="ignore")

# Impute: categorical -> mode, numeric -> median
cat_cols = df.select_dtypes(include="object").columns.tolist()
if "Loan_Status" in cat_cols:
    cat_cols.remove("Loan_Status")
num_cols = df.select_dtypes(include=np.number).columns.tolist()

for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

print("\n--- Missing values after cleaning ---")
print(df.isnull().sum().sum(), "missing values remaining")

## 3. ENCODE CATEGORICAL FEATURES

In [ ]:
df_encoded = df.copy()
encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    df_encoded[c] = le.fit_transform(df_encoded[c].astype(str))
    encoders[c] = le

target_le = LabelEncoder()
df_encoded["Loan_Status"] = target_le.fit_transform(df_encoded["Loan_Status"].astype(str))

print("\nCategorical columns encoded:", cat_cols)

## 4. TRAIN / TEST SPLIT + SCALING

In [ ]:
X = df_encoded.drop(columns=["Loan_Status"])
y = df_encoded["Loan_Status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. TRAIN + COMPARE MODELS

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
}

results = []
fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 4))

for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    results.append({"Model": name, "Accuracy": acc})

    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", ax=ax, cbar=False)
    ax.set_title(f"{name}\nAccuracy: {acc:.2%}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/loan_confusion_matrices.png", dpi=120)
plt.close()
print("Saved loan_confusion_matrices.png")

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False).reset_index(drop=True)
print("\n--- Model comparison ---")
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
print(f"\nBest model: {best_model_name}")
print("\n--- Classification report (best model) ---")
print(classification_report(y_test, best_model.predict(X_test_scaled), target_names=target_le.classes_))

print("\nDone. Files saved: loan_missing_values.png, loan_confusion_matrices.png")